# CASMI oracle fingerprint evaluation

MS-GPT generates candidate SMILES from the reference molecule's 4,096-bit Morgan fingerprint and the provided CASMI molecular formula. The notebook checks the first 0.1% of labeled rows, rounded down. Each spectrum counts once, so molecules with more spectra have more weight.

## Environment

Use the CUDA-enabled project environment and open this notebook from the project root. To launch Jupyter from PowerShell:

```powershell
& .\.venv\Scripts\python.exe -X utf8 -m jupyterlab
```

The run uses the labeled CASMI `train.parquet` and the [MS-GPT checkpoint, tokenizer, and source](SMILES_decoder_module/README.md). The decoder runs on CUDA with CUDA graph replay.

In [ ]:
from pathlib import Path

SAMPLES = 4
DECODER_BATCH_SIZE = 32
SEED = 42

ROOT = Path.cwd().resolve()
ARTIFACTS = ROOT / "spectrum_encoder_module" / "artifacts"
DATA = ARTIFACTS / "train.parquet"

In [ ]:
import pyarrow.parquet as pq

with pq.ParquetFile(DATA) as source:
    source_rows = source.metadata.num_rows

LIMIT = source_rows // 1000
if LIMIT < 1:
    raise ValueError("The 0.1% row limit rounds down to zero.")

OUTPUT_DIR = ARTIFACTS / f"pipeline_oracle_prefix_b{DECODER_BATCH_SIZE}_s{SAMPLES}_seed{SEED}_first_{LIMIT}_rows"
DATABASE = OUTPUT_DIR / "results.sqlite3"
print(f"Checking {LIMIT:,} of {source_rows:,} labeled rows. Results go to {DATABASE}")

## Evaluate

The pipeline saves one candidate pool per exact fingerprint and formula pair. Spectra with identical inputs share the same four draws. Each stage resumes from the last committed batch when run with the same settings and database.

In [ ]:
from spectrum_encoder_module.full_pipeline import run_pipeline

summary = run_pipeline(
    DATA,
    None,
    DATABASE,
    max_rows=LIMIT,
    samples=SAMPLES,
    batch_size=256,
    seed=SEED,
    compile_backend="eager",
    oracle_only=True,
    cuda_graph=True,
    prefix_attention=True,
    decoder_batch_size=DECODER_BATCH_SIZE,
)

## Results

Recovery checks 14-character InChIKey connectivity. Top-1 uses ranking by formula match and sample frequency. Decoder rates use rows with a valid reference and a formula supported by MS-GPT. Recovery across all rows includes skipped rows.

In [ ]:
import json

from IPython.display import Markdown, display

summary_path = OUTPUT_DIR / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")

print(f"Scanned {summary['rows_scanned']:,} rows and evaluated {summary['evaluated']:,}.")
print(f"Row status counts {summary['status_counts']}")
decode_seconds = summary["candidate_generation_seconds"]
if decode_seconds:
    inputs = summary["unique_decoder_inputs"]
    print(f"Decoded {inputs:,} unique inputs in {decode_seconds:.1f} s ({inputs / decode_seconds:.2f} inputs/s).")
print(f"Peak PyTorch CUDA allocation {summary['peak_cuda_allocated_mib']:,.0f} MiB")

metrics = {
    "Top-1 recovery": "top1_rate",
    f"Recovery in {SAMPLES} samples": "top_k_recovery_rate",
    "Formula match": "formula_match_rate",
    "Valid samples": "valid_sample_rate",
    "Recovery across all rows": "recovery_rate_all_rows",
}
table = ["| Metric | Oracle → MS-GPT |", "| :-- | --: |"]
for label, key in metrics.items():
    value = summary["arms"]["oracle"][key]
    rate = f"{value:.2%}" if value is not None else "—"
    table.append(f"| {label} | {rate} |")
display(Markdown("\n".join(table)))
print(f"Summary saved to {summary_path}")

In [ ]:
from contextlib import closing
import sqlite3

# Read one scored row to inspect the ranked SMILES from the oracle fingerprints.
with closing(sqlite3.connect(DATABASE)) as connection:
    example = connection.execute(
        "SELECT row_index, result FROM spectra WHERE status = ? ORDER BY row_index LIMIT 1",
        ("evaluated",),
    ).fetchone()

if example is not None:
    display({"parquet_row": example[0], **json.loads(example[1])})